# Consolidated Pipeline for statTableToText

This notebook aims to consolidate the pipeline for generating inferences from various large models and then checking those inferences. It uses the following models:
- Llama-3.1 70B
- Qwen-3 30B
- Gemma-4 26B

It follows three steps at the moment. Step 0 asks the models to generate tabular data that has 15 rows and 5 to 10 columns each. Step 1 asks the models to generate inferences based on the tabular data. Step 2 asks the models to check the statements for accuracy based on specific data. This is done by generating code to check over the statements.

Future work will include structured output in the JSON format in order to mitigate inference format generation for evaluation. Another thought that I had was to use a pipeline that does not rely on LLMs as code generating judges for more consistent accuracy. I am not sure if this is possible but that combined with not using LLM generated data could allow for us to isolate the inference ability of the LLM even further.

Install the necessary packages and configurate models

In [22]:
%pip install -U bitsandbytes huggingface_hub transformers accelerate

Note: you may need to restart the kernel to use updated packages.


In [23]:
from huggingface_hub import notebook_login
notebook_login()

In [24]:
from transformers import pipeline, GenerationConfig, AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch
import os

def generate_tables(model_key, model):
    
    prompt0 = [
        {"role": "user", "content": """You are a researcher familiar with statistical tables.
        Generate a random statistical table in CSV format which has the following characteristics:
    - The table should have between 5 and 10 columns.
    - The model can invent its own headers (both numeric and categorical).
    - Include 15 rows of data.
    - Values should be realistic and varied (integers, floats, categories, etc.).
    - Output only the table in CSV format, no explanations or extra text.
    - Do not generate more than one table.
    
    """},
    ]
    
    generation = generator(
        prompt0,
        generation_config=gen_config
    )
    
    print(f"Generation: {generation[0]['generated_text']}")
    
    # Get the assistant message from generated_text
    assistant_output = generation[-1]['generated_text'][-1]  # last item
    
    full_text = assistant_output['content']  # this is your CSV string
    
    # Preview
    print(full_text)
    
    # Folder where CSVs will be saved
    folder_path = "a.LLM_Tables"
    os.makedirs(folder_path, exist_ok=True)
    
    # Split the full string into candidate tables
    # We assume that each table starts with a header row (quotes optional)
    candidate_tables = full_text.strip().split("\n\n")
    #print(candidate_tables)
    
    table_counter = 1
    
    for table_text in candidate_tables:
        # Remove any trailing "assistant" or blank lines
        lines = [line for line in table_text.strip().split("\n") if line.strip() and line.strip().lower() != "assistant"]
    
        if not lines:
            continue  # skip empty sections
    
        header = lines[0]
    
        rows = lines[1:]
    
    
        # Check for incomplete table: all rows must have same number of columns as header
        num_cols = len(header.split(","))
        valid_rows = [r for r in rows if len(r.split(",")) == num_cols]
    
        if not valid_rows:
            continue  # skip tables with no valid rows
    
        # Reconstruct the table
        table_cleaned = "\n".join([header] + valid_rows)
    
        # Save to CSV
        file_name = f"{model_key}_table_{table_counter}.csv"
        full_path = os.path.join(folder_path, file_name)
        with open(full_path, "w", encoding="utf-8") as f:
            f.write(table_cleaned)
    
        print(f"Saved {file_name} with {len(valid_rows)} rows")
        table_counter += 1

In [25]:
def generate_inferences(model_key, model):
    import os

    folder_path = "a.LLM_Tables"
    folder_path_LLM_statements = "b.LLM_Inferences"
    os.makedirs(folder_path_LLM_statements, exist_ok=True)

    csv_files = [f for f in os.listdir(folder_path) if f.startswith(model_key) and f.endswith(".csv")]
    csv_files.sort()

    for csv_file in csv_files:
        full_path = os.path.join(folder_path, csv_file)

        with open(full_path, "r", encoding="utf-8") as f:
            table_text = f.read().strip()

        if not table_text:
            print(f"{csv_file} is empty, skipping")
            continue

        print(f"Processing {csv_file}")

        prompt1 = [
            {
                "role": "user",
                "content": f"""You are a data analyst tasked with generating clear, meaningful, and statistically-grounded inferences from a dataset.

Given the following table, generate 5-7 concise, distinct observations about the data. Follow these guidelines strictly:

**GUIDELINES:**

1. **Focus on Semantic Saliency**: Identify relationships between variables that make intuitive sense OR present surprising statistical correlations that contradict intuition.

2. **Avoid Redundancy**: Do not generate multiple statements expressing the same observation in different words. Each statement must convey new information.

3. **Avoid Undergeneralizations**: Do not add unnecessary conditions. For example, instead of "Every person who owns a car and has a pet works remotely," say "Most car owners work remotely" (if true and simpler).

4. **Avoid Vacuous Statements**: Only mention patterns that apply to a meaningful portion of the data (at least 50% of a category). Reject statements based on single cases or tiny subsets.

5. **Seek Probabilistic Saliency**: Highlight unexpected statistical relationships—patterns that are not semantically obvious but are strongly supported by the data.

6. **Use Clear Quantifiers**: Use phrases like "most," "many," "few," "all," "none," "the majority," "a minority," to express frequency and strength of relationships.

7. **Format**: Present each inference as a single, clear sentence. No bullet points, just numbered statements (1. 2. 3. etc.).

**TABLE:**
{table_text}

**YOUR TASK:**
Generate numbered inferences (1-7) that describe the most interesting, non-redundant patterns in this data. Each inference should be:
- Factually accurate (supported by the table)
- Meaningful and non-trivial
- Distinct from the others
- Written in clear, natural English

Do NOT include meta-commentary, explanations, or caveats. Only output the numbered inferences."""
            }
        ]

        generation = generator(
            prompt1,
            do_sample=False,
            temperature=1.0,
            top_p=1,
            max_new_tokens=10000,
            eos_token_id=tokenizer.eos_token_id
        )

        print(f"Generation: {generation[0]['generated_text']}")
        statements_LLM_output = generation[-1]["generated_text"][-1]
        statements_LLM_output_text = statements_LLM_output["content"]

        print(statements_LLM_output_text[:1000])

        file_name_LLM = f"{model_key}_inferences_{csv_file.split('_table_')[1][:-4]}.txt"
        full_path_LLM_statements = os.path.join(folder_path_LLM_statements, file_name_LLM)

        with open(full_path_LLM_statements, "w", encoding="utf-8") as f:
            f.write(statements_LLM_output_text)

        print(f"Saved {file_name_LLM}.")

In [26]:
def check_inferences(model_key, model):
    import re
    import subprocess
    import pandas as pd
    import os

    folder_path_LLM_statements = "b.LLM_Inferences"
    folder_path = "a.LLM_Tables"
    folder_path_python_code = "c.checking_statements"
    folder_path_results = "d.checking_statements_output"

    os.makedirs(folder_path_python_code, exist_ok=True)
    os.makedirs(folder_path_results, exist_ok=True)

    def extract_python_code(raw: str) -> str:
        match = re.search(r"```(?:python)?\s*\n(.*?)```", raw, re.DOTALL)
        if match:
            return match.group(1).strip()
        return raw.strip()

    LLM_statement_text_files = sorted(
        f for f in os.listdir(folder_path_LLM_statements)
        if f.startswith(model_key) and f.endswith(".txt")
    )

    csv_files = sorted(
        f for f in os.listdir(folder_path)
        if f.startswith(model_key) and f.endswith(".csv")
    )

    for LLM_statement_text_file in LLM_statement_text_files:
        full_path_stmt = os.path.join(folder_path_LLM_statements, LLM_statement_text_file)

        with open(full_path_stmt, "r", encoding="utf-8") as f:
            statements_text = f.read().strip()

        if not statements_text:
            print(f"{LLM_statement_text_file} is empty, skipping")
            continue

        print(f"\nProcessing {LLM_statement_text_file}.")

        stmt_num = LLM_statement_text_file.split("_inferences_")[1][:-4]

        matched_csv = None
        for csv_file in csv_files:
            csv_num = csv_file.split("_table_")[1][:-4]
            if csv_num == stmt_num:
                matched_csv = csv_file
                break

        if matched_csv is None:
            print(f"No matching CSV found for {LLM_statement_text_file}")
            continue

        full_csv_path = os.path.join(folder_path, matched_csv)
        df = pd.read_csv(full_csv_path)

        prompt2 = [
            {
                "role": "user",
                "content": f"""Do NOT repeat the instructions or the code provided.
Only output the requested Python code. Do NOT wrap the code in markdown fences.
Here's your task: Given the following statements:

{statements_text}

and the header names of the table:

{list(df.columns)}

write a python code (using pandas package) that checks whether each statement is True or False and prints a justification.
The CSV is already located at: "{full_csv_path}" and you should hardcode this path directly in the script.
It should also convert any numeric columns stored as strings into integers or floats when appropriate.
Everything you output must be valid, immediately runnable Python with no markdown or commentary outside comments.

Here is an example structure to follow:
import pandas as pd

def print_result(statement_no: int, description: str, truth: bool, explanation: str):
    status = "TRUE" if truth else "FALSE"
    print(f"\\nStatement {{statement_no}}: {{status}}")
    print(f"  - {{description}}")
    print(f"  - Explanation: {{explanation}}")

def main():
    df = pd.read_csv("{full_csv_path}")
    # add checks here

if __name__ == "__main__":
    main()"""
            }
        ]

        generation = generator(
            prompt2,
            do_sample=False,
            temperature=1.0,
            top_p=1,
            max_new_tokens=100000,
            eos_token_id=tokenizer.eos_token_id,
        )

        python_LLM_output = generation[-1]["generated_text"][-1]
        raw_code = python_LLM_output["content"]
        python_code = extract_python_code(raw_code)

        python_file_name_LLM = f"check_{model_key}_inferences_{stmt_num}.py"
        full_path_py = os.path.join(folder_path_python_code, python_file_name_LLM)

        with open(full_path_py, "w", encoding="utf-8") as f:
            f.write(python_code)
        print(f"Saved {python_file_name_LLM}")

        print(f"Running {python_file_name_LLM}...")
        result = subprocess.run(
            ["python3", full_path_py],
            capture_output=True,
            text=True,
        )

        if result.stdout:
            print(result.stdout)
        if result.returncode != 0:
            print(f"[ERROR] Script exited with code {result.returncode}")
            print(result.stderr)
        else:
            print(f"[OK] {python_file_name_LLM} completed successfully.")

        results_file_name = f"validation_{model_key}_inferences_{stmt_num}.txt"
        full_path_results_file = os.path.join(folder_path_results, results_file_name)

        with open(full_path_results_file, "w", encoding="utf-8") as f:
            f.write(result.stdout)
            if result.returncode != 0:
                f.write(f"\n[ERROR] Script exited with code {result.returncode}\n")
                f.write(result.stderr)

        print(f"Saved {results_file_name}")

In [27]:
def generate_master_summary(model_key, model):
    import os
    import re
    from datetime import datetime
    import pandas as pd
    
    # Folder paths
    folder_tables = "a.LLM_Tables"
    folder_inferences = "b.LLM_Inferences"
    folder_results = "d.checking_statements_output"
    folder_summary = "e.summary_reports"
    
    # Create summary folder if it doesn't exist
    os.makedirs(folder_summary, exist_ok=True)
    
    print(f"\nGenerating summaries for {model_key.upper()}...")
    
    # Get all validation result files for this model
    result_files = sorted(
        f for f in os.listdir(folder_results) if f.startswith(f"validation_{model_key}") and f.endswith(".txt")
    )
    
    if not result_files:
        print(f"No validation results found for {model_key}")
        return
    
    # Process each result file
    for result_file in result_files:
        # Extract table number from filename
        # e.g., "validation_llama_inferences_1.txt" -> table_num="1"
        match = re.search(r"_(\d+)\.txt$", result_file)
        if not match:
            continue
        
        table_num = match.group(1)
        table_file = f"{model_key}_table_{table_num}.csv"
        inference_file = f"{model_key}_inferences_{table_num}.txt"
        
        # Read validation results
        full_path_results = os.path.join(folder_results, result_file)
        with open(full_path_results, "r", encoding="utf-8") as f:
            validation_output = f.read()
        
        # Read inferences
        full_path_inferences = os.path.join(folder_inferences, inference_file)
        if not os.path.exists(full_path_inferences):
            continue
        
        with open(full_path_inferences, "r", encoding="utf-8") as f:
            inferences_text = f.read()
        
        # Read table
        full_path_table = os.path.join(folder_tables, table_file)
        if not os.path.exists(full_path_table):
            continue
        
        df = pd.read_csv(full_path_table)
        
        # Create table preview
        table_preview = df.head(3).to_string()
        table_shape = f"{df.shape[0]} rows × {df.shape[1]} columns"
        
        # Count inferences
        lines = inferences_text.split('\n')
        total_inferences = sum(1 for line in lines if re.match(r'^\d+\.\s', line.strip()))
        
        # Extract validation stats
        true_count = validation_output.count("TRUE")
        false_count = validation_output.count("FALSE")
        
        validation_stats = f"""  - TRUE statements: {true_count}
  - FALSE statements: {false_count}"""
        
        if true_count + false_count > 0:
            success_rate = 100*true_count/(true_count + false_count)
            validation_stats += f"\n  - Validation success rate: {true_count}/{true_count + false_count} ({success_rate:.1f}%)"
        
        # Generate the summary report
        summary_content = f"""
{'='*80}
COMPREHENSIVE INFERENCE SUMMARY REPORT
{'='*80}

Generated: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}

{'='*80}
EXPERIMENT CONFIGURATION
{'='*80}

Model: {model_key.upper()}
Table File: {table_file}
Table Shape: {table_shape}
Table Preview (first 3 rows):
{table_preview}

{'='*80}
PROMPTS USED
{'='*80}

1. TABLE GENERATION PROMPT:
   Task: Generate a random statistical table in CSV format
   - 5-10 columns
   - 15 rows of data
   - Mixed data types (numeric, categorical)
   - Realistic and varied values

2. INFERENCE GENERATION PROMPT:
   Task: Generate 5-7 statistical inferences from the table
   - Guidelines enforced:
     * Semantic saliency (intuitive or surprising patterns)
     * No redundancy
     * No undergeneralizations
     * Avoid vacuous statements (apply to ≥50% of data)
     * Clear quantifiers ("most", "many", "few", "all", "none")
   - Format: Numbered statements (1-7)

3. INFERENCE VALIDATION PROMPT:
   Task: Generate Python code to validate each inference
   - Check if statements are TRUE or FALSE
   - Provide explanations with evidence from data

{'='*80}
INFERENCES GENERATED
{'='*80}

{inferences_text}

{'='*80}
INFERENCE VALIDATION RESULTS
{'='*80}

{validation_output}

{'='*80}
SUMMARY STATISTICS
{'='*80}

Total Inferences: {total_inferences}

Validation Results:
{validation_stats}

{'='*80}
END OF REPORT
{'='*80}
"""
        
        # Save the summary
        summary_filename = f"summary_{model_key}_table_{table_num}.txt"
        full_path_summary = os.path.join(folder_summary, summary_filename)
        
        with open(full_path_summary, "w", encoding="utf-8") as f:
            f.write(summary_content)
        
        print(f"  ✓ {summary_filename}")

In [28]:
import gc
import time
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

# Configure 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    llm_int8_enable_fp32_cpu_offload=True,
)
 
# Model definitions
models_info = {
    "gemma": "google/gemma-2-27b-it",
    "qwen": "Qwen/Qwen2.5-32B-Instruct",
    "llama": "meta-llama/Llama-3.1-70B-Instruct",
}
 
# Loop through all models
for model_key, model_name in models_info.items():
    print(f"\n{'='*70}")
    print(f"PROCESSING: {model_key.upper()}")
    print(f"Model: {model_name}")
    print(f"{'='*70}\n")
    
    # Load the model
    print(f"Loading {model_key}...")
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map="auto",
        low_cpu_mem_usage=True,
        offload_buffers=True,
    )
    
    print(f"✓ {model_key.upper()} loaded")
    print(f"  CUDA Memory: {torch.cuda.memory_allocated() / 1e9:.1f}GB / {torch.cuda.memory_reserved() / 1e9:.1f}GB\n")

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    generator = pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer
    )
    
    gen_config = GenerationConfig(
        do_sample=False,
        temperature=1.0,
        top_p=1.0,
        max_new_tokens=1000,
        eos_token_id=None
    )
        
    # Generate tables
    generate_tables(model_key, model)

    # Generate inferences
    generate_inferences(model_key, model)

    # Check inferences
    check_inferences(model_key, model)

    # Generate summary
    generate_master_summary(model_key, model)
    
    
    # Clear memory before next model
    print(f"\nClearing {model_key} from VRAM...")
    del model
    gc.collect()
    torch.cuda.empty_cache()
    time.sleep(1)  # Give CUDA time to clean up
    print(f"✓ Memory cleared\n")
 
print(f"\n{'='*70}")
print("ALL MODELS PROCESSED SUCCESSFULLY")
print(f"{'='*70}")


PROCESSING: GEMMA
Model: google/gemma-2-27b-it

Loading gemma...


Loading weights:   0%|          | 0/508 [00:00<?, ?it/s]

✓ GEMMA loaded
  CUDA Memory: 82.9GB / 83.7GB



Both `max_new_tokens` (=10000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Generation: [{'role': 'user', 'content': 'You are a researcher familiar with statistical tables.\n        Generate a random statistical table in CSV format which has the following characteristics:\n    - The table should have between 5 and 10 columns.\n    - The model can invent its own headers (both numeric and categorical).\n    - Include 15 rows of data.\n    - Values should be realistic and varied (integers, floats, categories, etc.).\n    - Output only the table in CSV format, no explanations or extra text.\n    - Do not generate more than one table.\n\n    '}, {'role': 'assistant', 'content': "Age,Gender,Income,Education,City,Pet,Satisfaction,Purchase Frequency,Loyalty Program,Feedback\n28,Male,55000,Bachelor's,New York,Dog,High,Monthly,Yes,Positive\n35,Female,72000,Master's,Los Angeles,Cat,Medium,Weekly,No,Neutral\n42,Male,90000,PhD,Chicago,None,Low,Yearly,Yes,Negative\n22,Female,38000,Associate's,Miami,Bird,High,Monthly,No,Positive\n51,Male,110000,Master's,Houston,Dog,Medium,Qu

...

✓ LLAMA loaded
  CUDA Memory: 74.7GB / 97.3GB



Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Both `max_new_tokens` (=10000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Generation: [{'role': 'user', 'content': 'You are a researcher familiar with statistical tables.\n        Generate a random statistical table in CSV format which has the following characteristics:\n    - The table should have between 5 and 10 columns.\n    - The model can invent its own headers (both numeric and categorical).\n    - Include 15 rows of data.\n    - Values should be realistic and varied (integers, floats, categories, etc.).\n    - Output only the table in CSV format, no explanations or extra text.\n    - Do not generate more than one table.\n\n    '}, {'role': 'assistant', 'content': '"City","Population","Region","GDP per Capita","Unemployment Rate","Average Temperature","Education Level","Industry","Transportation","Housing Cost"\n"New York",8405837,"Northeast",69353,4.2,12.3,"Bachelor\'s degree",Finance,"Subway",3500\n"Los Angeles",3990456,"West",58392,4.5,18.1,"Some college",Entertainment,"Driving",2800\n"Chicago",2695598,"Midwest",57494,5.1,8.9,"High school",Manufact

Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Both `max_new_tokens` (=100000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Generation: [{'role': 'user', 'content': 'You are a data analyst tasked with generating clear, meaningful, and statistically-grounded inferences from a dataset.\n\nGiven the following table, generate 5-7 concise, distinct observations about the data. Follow these guidelines strictly:\n\n**GUIDELINES:**\n\n1. **Focus on Semantic Saliency**: Identify relationships between variables that make intuitive sense OR present surprising statistical correlations that contradict intuition.\n\n2. **Avoid Redundancy**: Do not generate multiple statements expressing the same observation in different words. Each statement must convey new information.\n\n3. **Avoid Undergeneralizations**: Do not add unnecessary conditions. For example, instead of "Every person who owns a car and has a pet works remotely," say "Most car owners work remotely" (if true and simpler).\n\n4. **Avoid Vacuous Statements**: Only mention patterns that apply to a meaningful portion of the data (at least 50% of a category). Reject